In [ ]:
from src.clp_zne.qpu_info.layout_cycles import heron_layout_cycles_12q
from src.clp_zne.hamiltonians import sherrington_kirkpatrick_model
from src.clp_zne.qpu_info.backends import FakeTorino
from src.clp_zne.mitigate import clp_zne_mitigate_general_topology_circuit
from src.clp_zne.backends import NClusterBackend
from src.clp_zne.utils import compute_evals_ideal, get_grid_entanglement

from qiskit.circuit.library import TwoLocal

import os
import numpy as np
from tqdm import tqdm

In [ ]:
OUTPUT_FOLDER = r"data\NClusterBackend--Square grid circuit--9 qubits--Orig noise--CLP ZNE 1 param 3 cycles"

ROWS, COLS = 3, 3
N_QUBITS = ROWS * COLS
N_CIRCUITS = 20
N_LAYERS = 3
N_OBSERVABLES = 100
T1_T2_NOISE_MULTIPLIER = 1
SEED = 42

In [ ]:
source_backend = FakeTorino()
selected_layouts = [heron_layout_cycles_12q[i] for i in [5, 9, 13]]

backend = NClusterBackend.from_backend(
                cluster_size=N_QUBITS, 
                source_backend=source_backend,
                list_of_index_groups=selected_layouts, 
                seed=SEED
            )

In [ ]:
# Define circuit connectivity and ansatz
entanglement_map = get_grid_entanglement(ROWS, COLS)
ansatz = TwoLocal(N_QUBITS, ['rx', 'rz'], 'cz', entanglement=entanglement_map, reps=N_LAYERS)

# Generate circuits
circuits = []
for i in range(N_CIRCUITS):
    rng = np.random.default_rng(i)
    circuit = ansatz.copy()
    parameters = rng.uniform(-np.pi, np.pi, circuit.num_parameters)
    circuit.assign_parameters(parameters, inplace=True)
    circuits.append(circuit)

# Define observables
observables = [sherrington_kirkpatrick_model(N_QUBITS, h=1, seed=i) for i in range(N_OBSERVABLES)]

In [ ]:
all_evals_ideal = []
all_evals_mitigated = []
all_evals_noisy = []
all_error_sums = []

# Initial circuit layouts in different qubit cycles
layouts_for_cyclic_permutations = np.arange(backend.num_qubits).reshape((backend.num_clusters, -1)).tolist()

for circ in tqdm(circuits):
    # Perform error mitigation
    evals_mitigated, evals_noisy, error_sums = clp_zne_mitigate_general_topology_circuit(circ, observables, 
                                                                        layouts_for_cyclic_permutations, 
                                                                        backend, num_params=1,
                                                                        therm_noise_multiplier=T1_T2_NOISE_MULTIPLIER)
    # Compute noiseless expectation value
    evals_ideal = compute_evals_ideal(circ, observables)
    
    all_evals_ideal.append(evals_ideal)
    all_evals_mitigated.append(evals_mitigated)
    all_evals_noisy.append(evals_noisy)
    all_error_sums.append(error_sums)

In [ ]:
# Save the results
data_to_save = {
    'evals_ideal.npy': np.array(all_evals_ideal),
    'evals_mitigated.npy': np.array(all_evals_mitigated),
    'evals_noisy.npy': np.array(all_evals_noisy),
    'error_sums.npy': np.array(all_error_sums)
}

for filename, data in data_to_save.items():
    path = os.path.join(OUTPUT_FOLDER, filename)
    np.save(path, data)

print(f"Successfully saved {len(data_to_save)} files to: {OUTPUT_FOLDER}")